# 10 Python intermediate - metaklasy
_Kamil Bartocha_ | _wersja 2.0_

## Rozkład jazdy

1. 🔹 **Metaklasy i `type`** - klasa klas, `type()` jako fabryka
2. 🔹 **Tworzenie metaklasy** - `__new__`, walidacja, auto-dodawanie
3. 🔹 **Praktyczne wzorce** - rejestr, logging metod, Singleton
4. 🔹 **`__init_subclass__`** - nowoczesna alternatywa dla metaklas

🐍 Każda sekcja zawiera ćwiczenia.

## 1. 🔹 Metaklasy i `type`

W Pythonie wszystko jest obiektem - w tym klasy. Klasa jest instancją
swojej *metaklasy*. Domyślną metaklasą wszystkich klas jest `type`.

```
obiekt     <- instancja <- klasa
klasa      <- instancja <- metaklasa (type)
```

`type` pełni podwójną rolę:
- `type(obj)` - zwraca typ/klasę obiektu
- `type(name, bases, dct)` - *tworzy* nową klasę dynamicznie

| Wywołanie | Co robi |
|-----------|--------|
| `type(obj)` | zwraca klasę obiektu |
| `type(MyClass)` | zwraca metaklasę klasy |
| `type('Foo', (Base,), {'x': 1})` | tworzy klasę `Foo` |

> 💡 **Tip:** Metaklasy to zaawansowany mechanizm, rzadko potrzebny
> w codziennym kodzie. Warto je znać, bo używają ich popularne
> frameworki (Django ORM, SQLAlchemy, ABC).

In [ ]:
# type() - inspekcja
class Dog:
    pass

rex = Dog()

print(type(rex))       # <class '__main__.Dog'>
print(type(Dog))       # <class 'type'>
print(type(int))       # <class 'type'>
print(type(type))      # <class 'type'>  (type jest swoją własną metaklasą)

# isinstance vs type
print(isinstance(rex, Dog))    # True
print(isinstance(Dog, type))   # True  (klasa jest instancją type)

In [ ]:
# type() jako fabryka klas - dynamiczne tworzenie
# type(name, bases, dict) == class name(bases): dict

# Rownowazne zapisy:

# Sposob 1: standardowy
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"Point({self.x}, {self.y})"


# Sposob 2: przez type()
def point_init(self, x, y):
    self.x = x
    self.y = y

def point_str(self):
    return f"Point({self.x}, {self.y})"

DynPoint = type('DynPoint', (), {
    '__init__': point_init,
    '__str__':  point_str,
})

p = DynPoint(3, 4)
print(p)                       # Point(3, 4)
print(type(p).__name__)        # DynPoint

---

### 🐍 Ćwiczenia - Metaklasy i `type`

**Ćw. 1.** Sprawdź metaklasę kilku wbudowanych typów:
`int`, `str`, `list`, `dict`. Czy każdy z nich jest instancją `type`?

**Ćw. 2.** Używając `type()` jako fabryki, stwórz dynamicznie klasę
`Rectangle` z atrybutem `default_color = 'red'` i metodą
`area(self)` zwracającą `self.width * self.height`.

**Ćw. 3. *(Trudniejsze)*** Napisz funkcję `make_class(name, attrs)`,
która przyjmuje nazwę i słownik atrybutów i zwraca nową klasę
z tymi atrybutami. Sprawdź że dziedziczenie po `object` działa.

In [ ]:
# Ćw. 1: metaklasy wbudowanych typow
for typ in [int, str, list, dict]:
    print(f"type({typ.__name__}) = {type(typ)}")

# Czy sa instancjami type?
for typ in [int, str, list]:
    print(f"isinstance({typ.__name__}, type) = {isinstance(typ, type)}")

In [ ]:
# Ćw. 2: Rectangle przez type()
def rect_init(self, width, height):
    self.width  = width
    self.height = height

def rect_area(self):
    ...


Rectangle = type('Rectangle', (object,), {
    'default_color': 'red',
    '__init__': rect_init,
    'area':     ...,
})


r = Rectangle(4, 6)
print(r.area())              # 24
print(r.default_color)       # red

In [ ]:
# Ćw. 3: make_class
def make_class(name, attrs):
    ...


Config = make_class('Config', {'debug': True, 'port': 8080})
c = Config()
print(c.debug)          # True
print(c.port)           # 8080
print(type(c).__name__) # Config
print(isinstance(c, object))  # True

## 2. 🔹 Tworzenie metaklasy

Własną metaklasę tworzymy przez dziedziczenie po `type`.
Kluczowa metoda to `__new__`, która jest wywoływana *podczas
definicji klasy* (nie podczas tworzenia instancji).

```python
class MyMeta(type):
    def __new__(cls, name, bases, dct):
        # name  - napis: nazwa definiowanej klasy
        # bases - tupla: klasy bazowe
        # dct   - słownik: atrybuty i metody
        return super().__new__(cls, name, bases, dct)

class MyClass(metaclass=MyMeta):
    pass
```

Dwa główne zastosowania `__new__`:

| Cel | Co robimy w `__new__` |
|-----|----------------------|
| Walidacja | sprawdź `dct`, zgłoś `TypeError` jeśli warunek niespełniony |
| Auto-dodawanie | dodaj klucze do `dct` przed wywołaniem `super().__new__` |

> ⚠️ Pamiętaj o `return super().__new__(cls, name, bases, dct)` -
> bez tego klasa nie zostanie utworzona.

In [ ]:
# Walidacja - wymuszenie obecnosci metody
class RequireMethodMeta(type):
    def __new__(cls, name, bases, dct):
        if bases and 'process' not in dct:
            raise TypeError(
                f"Class '{name}' must define 'process()'"
            )
        return super().__new__(cls, name, bases, dct)


class Processor(metaclass=RequireMethodMeta):
    pass   # base class - pomijamy walidacje (no bases)


class GoodProcessor(Processor):
    def process(self, data):
        return data.upper()


try:
    class BadProcessor(Processor):
        pass   # TypeError: must define 'process()'
except TypeError as e:
    print(e)

In [ ]:
# Auto-dodawanie atrybutow i metod
class AutoDescribeMeta(type):
    def __new__(cls, name, bases, dct):
        if 'describe' not in dct:
            def describe(self):
                attrs = {k: v for k, v in vars(self).items()}
                return f"{name}: {attrs}"
            dct['describe'] = describe

        if 'class_name' not in dct:
            dct['class_name'] = name

        return super().__new__(cls, name, bases, dct)


class Person(metaclass=AutoDescribeMeta):
    def __init__(self, name, age):
        self.name = name
        self.age  = age


p = Person('Alice', 30)
print(p.describe())      # Person: {'name': 'Alice', 'age': 30}
print(p.class_name)      # Person

---

### 🐍 Ćwiczenia - Tworzenie metaklasy

**Ćw. 4.** Napisz metaklasę `UpperAttrsMeta`, która wymusza, żeby
wszystkie atrybuty (klasy i instancji) - poza dunder-ami - były
zdefiniowane z nazwami WIELKIMI_LITERAMI. Jeśli nie - `TypeError`.

**Ćw. 5.** Napisz metaklasę `VersionedMeta`, która automatycznie
dodaje atrybut `version = '1.0'` do każdej klasy, która go nie
zdefiniuje, oraz atrybut `created_by = 'VersionedMeta'`.

**Ćw. 6. *(Trudniejsze)*** Napisz metaklasę `ReadOnlyMeta`, która
uniemożliwia zmianę wartości atrybutów klasy po jej zdefiniowaniu
(nadpisz `__setattr__` w metaklasie).

In [ ]:
# Ćw. 4: UpperAttrsMeta
class UpperAttrsMeta(type):
    def __new__(cls, name, bases, dct):
        for key in dct:
            if not key.startswith('__') and not key.isupper():
                raise TypeError(
                    f"Attribute '{key}' in class '{name}' "
                    f"must be UPPERCASE"
                )
        return ...


class Config(metaclass=UpperAttrsMeta):
    HOST = 'localhost'
    PORT = 8080

print(Config.HOST)   # localhost

try:
    class BadConfig(metaclass=UpperAttrsMeta):
        host = 'localhost'   # TypeError
except TypeError as e:
    print(e)

In [ ]:
# Ćw. 5: VersionedMeta
class VersionedMeta(type):
    def __new__(cls, name, bases, dct):
        if 'version' not in dct:
            dct['version'] = ...
        dct['created_by'] = ...
        return super().__new__(cls, name, bases, dct)


class Alpha(metaclass=VersionedMeta):
    pass

class Beta(metaclass=VersionedMeta):
    version = '2.5'


print(Alpha.version)     # 1.0
print(Beta.version)      # 2.5
print(Alpha.created_by)  # VersionedMeta
print(Beta.created_by)   # VersionedMeta

In [ ]:
# Ćw. 6: ReadOnlyMeta
class ReadOnlyMeta(type):
    def __setattr__(cls, name, value):
        # hint: sprawdz czy klasa juz istnieje (hasattr(cls, name))
        if hasattr(cls, name):
            raise AttributeError(
                f"Cannot modify read-only attribute '{name}'"
            )
        ...


class Constants(metaclass=ReadOnlyMeta):
    PI    = 3.14159
    SPEED = 299_792_458


print(Constants.PI)   # 3.14159

try:
    Constants.PI = 3   # AttributeError
except AttributeError as e:
    print(e)

## 3. 🔹 Praktyczne wzorce

Trzy wzorce, w których metaklasy sprawdzają się najlepiej:

In [ ]:
# Wzorzec 1: Rejestr klas
class RegistryMeta(type):
    registry = {}

    def __new__(cls, name, bases, dct):
        new_class = super().__new__(cls, name, bases, dct)
        cls.registry[name] = new_class
        return new_class


class Plugin(metaclass=RegistryMeta):
    pass

class AuthPlugin(Plugin):
    pass

class LogPlugin(Plugin):
    pass


print(RegistryMeta.registry.keys())
# dict_keys(['Plugin', 'AuthPlugin', 'LogPlugin'])

In [ ]:
import functools

# Wzorzec 2: Auto-logowanie wszystkich metod
def log_call(func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        print(f"  call: {func.__name__}({args}, {kwargs})")
        return func(self, *args, **kwargs)
    return wrapper


class LogAllMethodsMeta(type):
    def __new__(cls, name, bases, dct):
        for attr, val in dct.items():
            if callable(val) and not attr.startswith('__'):
                dct[attr] = log_call(val)
        return super().__new__(cls, name, bases, dct)


class Calculator(metaclass=LogAllMethodsMeta):
    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b


calc = Calculator()
print(calc.add(3, 4))       # call: add((3, 4), {}) -> 7
print(calc.multiply(2, 5))  # call: multiply((2, 5), {}) -> 10

In [ ]:
# Wzorzec 3: Singleton
# Singleton - zawsze zwraca te sama instancje
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]


class AppConfig(metaclass=SingletonMeta):
    def __init__(self):
        self.debug = False
        self.port  = 8080


a = AppConfig()
b = AppConfig()

print(a is b)     # True - ta sama instancja
print(id(a) == id(b))   # True

a.debug = True
print(b.debug)    # True - to ten sam obiekt

---

### 🐍 Ćwiczenia - Praktyczne wzorce

**Ćw. 7.** Rozbuduj wzorzec rejestru: napisz metaklasę `TypeRegistryMeta`,
która rejestruje tylko klasy bezpośrednio dziedziczące po klasie bazowej
(pomiń samą klasę bazową).

**Ćw. 8.** Napisz metaklasę `CountInstancesMeta`, która dodaje
do każdej klasy atrybut klasowy `instance_count = 0` i automatycznie
go inkrementuje przy każdym wywołaniu `__init__`.

**Ćw. 9. *(Trudniejsze)*** Napisz metaklasę `AbstractMeta`, która
uniemożliwia tworzenie instancji klas oznaczonych atrybutem
`abstract = True`. Klasy pochodne (bez tego atrybutu) mogą być
instancjonowane normalnie.

In [ ]:
# Ćw. 7: TypeRegistryMeta - rejestruje tylko podklasy
class TypeRegistryMeta(type):
    registry = {}

    def __new__(cls, name, bases, dct):
        new_class = super().__new__(cls, name, bases, dct)
        # hint: bases != () oznacza, ze to nie jest klasa bazowa
        if bases:
            ...
        return new_class


class Animal(metaclass=TypeRegistryMeta):
    pass

class Dog(Animal): pass
class Cat(Animal): pass


print(TypeRegistryMeta.registry)   # {'Dog': ..., 'Cat': ...}  (nie 'Animal')

In [ ]:
# Ćw. 8: CountInstancesMeta
class CountInstancesMeta(type):
    def __new__(cls, name, bases, dct):
        dct['instance_count'] = 0
        original_init = dct.get('__init__')

        def __init__(self, *args, **kwargs):
            type(self).instance_count += 1
            if original_init:
                original_init(self, *args, **kwargs)

        dct['__init__'] = __init__
        return ...


class Car(metaclass=CountInstancesMeta):
    pass


Car()
Car()
Car()
print(Car.instance_count)   # 3

In [ ]:
# Ćw. 9: AbstractMeta
class AbstractMeta(type):
    def __call__(cls, *args, **kwargs):
        if cls.__dict__.get('abstract', False):
            raise TypeError(
                f"Cannot instantiate abstract class '{cls.__name__}'"
            )
        return ...


class Shape(metaclass=AbstractMeta):
    abstract = True


class Circle(Shape):
    def __init__(self, r):
        self.r = r


c = Circle(5)
print(c.r)    # 5

try:
    s = Shape()   # TypeError
except TypeError as e:
    print(e)

## 4. 🔹 `__init_subclass__` - nowoczesna alternatywa

Od Python 3.6 wiele przypadków użycia metaklas można zastąpić
prostszą metodą `__init_subclass__`. Jest wywoływana automatycznie
za każdym razem gdy jakaś klasa dziedziczy po danej klasie bazowej.

```python
class Base:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        # cls to definiowana podklasa
```

| Cel | Metaklasa | `__init_subclass__` |
|-----|-----------|---------------------|
| Walidacja podklasy | `__new__` w metaklasie | `__init_subclass__` |
| Rejestr klas | `__new__` w metaklasie | `__init_subclass__` |
| Modyfikacja klasy bazowej | metaklasa | nie dotyczy |
| Singleton / `__call__` | metaklasa | nie dotyczy |

> 💡 **Tip:** Jeśli chcesz tylko zareagować na dziedziczenie -
> użyj `__init_subclass__`. Metaklasę wybierz gdy potrzebujesz
> kontroli nad samym procesem tworzenia klasy (`__call__`, zmiana
> `type` w hierarchii).

In [ ]:
# __init_subclass__ zamiast metaklasy - rejestr
class Plugin:
    _registry = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin._registry[cls.__name__] = cls


class AuthPlugin(Plugin):
    pass

class LogPlugin(Plugin):
    pass


print(Plugin._registry.keys())
# dict_keys(['AuthPlugin', 'LogPlugin'])

In [ ]:
# __init_subclass__ z parametrami (przez keyword args)
class Animal:
    def __init_subclass__(cls, sound='...', **kwargs):
        super().__init_subclass__(**kwargs)
        cls.sound = sound


class Dog(Animal, sound='Woof'):
    pass

class Cat(Animal, sound='Meow'):
    pass

class Fish(Animal):
    pass


print(Dog.sound)    # Woof
print(Cat.sound)    # Meow
print(Fish.sound)   # ...

---

### 🐍 Ćwiczenia - `__init_subclass__`

**Ćw. 10.** Użyj `__init_subclass__` w klasie `Command`, żeby
automatycznie rejestrować każdą podklasę w słowniku
`Command.commands` (klucz = nazwa klasy małymi literami).

**Ćw. 11.** Użyj `__init_subclass__` w klasie `Validator`, żeby
wymuszać obecność metody `validate(self, value)` w każdej podklasie.
Jeśli brak - zgłoś `TypeError`.

**Ćw. 12. *(Trudniejsze)*** Porównaj implementacje: napisz to samo
zachowanie (rejestracja + walidacja) raz przez metaklasę i raz przez
`__init_subclass__`. Użyj `abc.ABC` jako przykładu bazowego.

In [ ]:
# Ćw. 10: Command - rejestr przez __init_subclass__
class Command:
    commands = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        ...


class HelpCommand(Command):
    pass

class QuitCommand(Command):
    pass


print(Command.commands)
# {'helpcommand': <class 'HelpCommand'>, 'quitcommand': <class 'QuitCommand'>}

In [ ]:
# Ćw. 11: Validator z walidacja metody
class Validator:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        if 'validate' not in cls.__dict__:
            raise TypeError(
                f"'{cls.__name__}' must implement validate()"
            )


class EmailValidator(Validator):
    def validate(self, value):
        return '@' in value


print(EmailValidator().validate('alice@example.com'))   # True

try:
    class BadValidator(Validator):
        pass   # TypeError
except TypeError as e:
    print(e)

In [ ]:
import abc

# Ćw. 12: Porownanie metaklasa vs __init_subclass__

# Podejście 1: metaclass
class PluginMeta(type):
    registry = {}
    def __new__(cls, name, bases, dct):
        if bases and 'run' not in dct:
            raise TypeError(f"'{name}' must implement run()")
        new_cls = super().__new__(cls, name, bases, dct)
        if bases:
            cls.registry[name] = new_cls
        return new_cls

class BasePlugin(metaclass=PluginMeta): pass
class EmailPlugin(BasePlugin):
    def run(self): return 'email'


# Podejście 2: __init_subclass__ (prostsze)
class Plugin2:
    registry = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        if 'run' not in cls.__dict__:
            raise TypeError(f"'{cls.__name__}' must implement run()")
        Plugin2.registry[cls.__name__] = cls

class LogPlugin2(Plugin2):
    def run(self): return 'log'


print(PluginMeta.registry)   # {'EmailPlugin': ...}
print(Plugin2.registry)      # {'LogPlugin2': ...}